# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'https://raw.githubusercontent.com/jasontpv/DS2002FA26/refs/heads/main/notebooks/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop = cat[cat['product_name'].str.lower().str.contains('pop')]

print('rows that look like Pop-Tart products:', len(pop))
pop

rows that look like Pop-Tart products: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


Explanation: I lowercased the product names and filtered for the ones containing pop, which found 4 rows that are all strawberry Pop-Tarts from four different formats.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
is_strawberry = pop['product_name'].str.lower().str.contains('strawberry')

strawberry = pop[is_strawberry]
not_strawberry = pop[~is_strawberry]

print('strawberry Pop-Tart SKUs:', len(strawberry))
print(strawberry[['sku', 'product_name']].to_string(index=False))
print()
print('Pop-Tart rows that are not strawberry:', len(not_strawberry))

strawberry Pop-Tart SKUs: 4
          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry
PT_STRAWBERRY pop-tarts strawberry

Pop-Tart rows that are not strawberry: 0


**Pop-Tart rows that are a different product:** _None. All 4 rows that matched are strawberry, so there is no other flavor in this catalog that needs to stay separate._

Explanation: I split the Pop-Tart rows into strawberry and not strawberry, which showed all 4 are strawberry under different SKU formats and none are a different flavor.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
CANON = {
    'POPTART-STRAW':  'POPTART-STRAWBERRY',
    'PT-12':          'POPTART-STRAWBERRY',
    'SKU#459812':     'POPTART-STRAWBERRY',
    'PT_STRAWBERRY':  'POPTART-STRAWBERRY',
}

cat['canonical_sku'] = cat['sku'].map(CANON).fillna(cat['sku'])

print('rows:', len(cat))
print('nulls in canonical_sku:', cat['canonical_sku'].isna().sum())
print('distinct SKUs before:', cat['sku'].nunique(), 'after:', cat['canonical_sku'].nunique())
print(cat[cat['canonical_sku'] == 'POPTART-STRAWBERRY'][['sku', 'product_name', 'canonical_sku']].to_string(index=False))

rows: 45
nulls in canonical_sku: 0
distinct SKUs before: 45 after: 42
          sku         product_name      canonical_sku
        PT-12 pop-tarts strawberry POPTART-STRAWBERRY
   SKU#459812  Strawberry Pop-Tart POPTART-STRAWBERRY
POPTART-STRAW Pop-Tarts Strawberry POPTART-STRAWBERRY
PT_STRAWBERRY pop-tarts strawberry POPTART-STRAWBERRY


Explanation: I made a dictionary pointing all four strawberry Pop-Tart SKUs at one canonical SKU called POPTART-STRAWBERRY, then used map with fillna so every SKU not in the dictionary keeps its own code instead of becoming NaN. That took the catalog from 45 distinct SKUs down to 42 and created no nulls.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print('before:', cat['category'].nunique(), 'distinct')
print(cat['category'].value_counts().to_string())

cat['category'] = cat['category'].str.strip().str.title()

print('after:', cat['category'].nunique(), 'distinct')
print(cat['category'].value_counts().to_string())

before: 27 distinct
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
after: 15 distinct
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods   

Explanation: I stripped whitespace and title-cased the category column, which cut the distinct values from 27 down to 15 by merging the capitalization differences. Five pairs are still two names for one category, like Grocery and Groceries.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
cat['normalized'] = (cat['product_name'].str.lower()
                     .str.replace(r'[^a-z ]', '', regex=True)
                     .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (cat.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
print(candidates.to_string())

grouped = {s for group in candidates for s in group}
alone = cat[~cat['sku'].isin(grouped)]

print('SKUs the template left alone:', len(alone))
print(alone[['sku', 'product_name']].to_string(index=False))

normalized
aa batteries pack                     [BAT-AA4, BATT_AA_4PK]
beer light pack                          [BEER-24, LTBEER24]
board game monopoly        [BG-MONO, MONOPOLY-01, GAME-MONO]
bread white                           [BREAD-WH, WHITEBREAD]
butter peanut                            [PB-16, PNUTBTR-16]
candles emergency                     [CNDL-EM, CANDLE-EMRG]
canned soup                            [SOUP-CAN, CNSOUP-01]
cards playing                       [CARDS-01, PLAYCARD-STD]
chips pack variety                   [CHIPS-VP, CHIPVARIETY]
duty heavy tarp                         [TARP-HD, HDTARP-01]
flashlight led                         [FLASH-01, LED-FLASH]
paper towels                         [PTOWEL-6, PAPERTOWEL6]
poptarts strawberry    [PT-12, POPTART-STRAW, PT_STRAWBERRY]
SKUs the template left alone: 17
          sku           product_name
   SKU#459812    Strawberry Pop-Tart
       WTR-24     BOTTLED WATER 24PK
    WATER24PK  Bottled Water 24-Pack
     WTR_24PK  

Explanation: I used the template from the lecture, which lowercases each product name, takes out punctuation and numbers, and sorts the words so the same words in a different order end up matching. It found 13 groups, so I added a few lines to print the 17 SKUs it left alone, which is where the fragmentation it missed shows up, like the three bottled waters and the fourth Pop-Tart SKU.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat['canonical_sku'].notna().all(), 'consolidation created nulls'
assert len(cat) == 45, 'row count changed'
assert cat['canonical_sku'].nunique() <= cat['sku'].nunique(), 'consolidation added SKUs'
print('mapping is safe: no nulls, no rows added or lost.')

mapping is safe: no nulls, no rows added or lost.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Monopoly is split across three SKUs, BG-MONO, MONOPOLY-01, and GAME-MONO, two called "Monopoly Board Game" and one called "Board Game - Monopoly". Light beer is split across two, BEER-24 and LTBEER24, which have the exact same product name of "Light Beer (24-pack)" and differ only in the SKU code. I group them both the same way I did the Pop-Tarts, by picking one canonical SKU and writing a dictionary.

b) It becomes a business decision once you are merging things that are not clearly the same item. Cleaning is when two names clearly point at one product, like Monopoly Board Game and Board Game - Monopoly, but it stops being cleaning when you have to decide whether two similar products are really one. For example, I am unsure about CHRG-PORT, called "PORTABLE CHARGER", and PORTCHARGER, called "Portable Phone Charger", which could be the same thing or could be a general charger and one meant specifically for phones. I would ask a manager since they would know whether those are two things the store orders separately.